In [ ]:
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI
import os
import operator
from langgraph.checkpoint.memory import MemorySaver


In [ ]:
load_dotenv()
api_key = os.getenv('GOOGLE_API_KEY')

model = ChatGoogleGenerativeAI(
    model = 'gemini-3.8-flash',
    api_key = api_key
)

In [ ]:


from langgraph.graph import add_messages


class ChatState(TypedDict):

    messages: Annotated[list[BaseMessage], add_messages ]

In [ ]:
def chat_node(state: ChatState):

    messages = state['messages']

    response = model.invoke(messages)

    return {'messages': [response]}

In [ ]:
checkpointer = MemorySaver()
graph = StateGraph(ChatState)

graph.add_node('chat_node',chat_node)

graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node', END)

chatbot = graph.compile(checkpointer = checkpointer)

In [ ]:
# wha

In [ ]:
thread_id = '1'

while True:

    user_message = input('Type here: ')

    print('User: ', user_message)
    if user_message.strip().lower() in ['exit', 'quit', 'bye']:
        break

    config = {'configurable': { 'thread_id': thread_id}}
    response = chatbot.invoke({'messages': [HumanMessage(content=user_message)]}, config=config)

    print('AI: ', response['messages'][-1].content)

User:  hiii
AI:  [{'type': 'text', 'text': "Hiii! How's it going? What's on your mind today?", 'extras': {'signature': 'EsgGCsUGAWkUfRM2QBb1LWO8rpMRqNrAis1mZnZ8H8as7Ofddfb/+elQUWey+Xn2uG8EMMcnNv7FzzG+2dNWNTm+WH4SGHPmXp8NqWvt6i6B3NqOsT2NcVelvIkIJILm3Jdx1xG706K1tU+ZfzXR/1hg6w/8qkrlq2f/3pt9STa4Qp1q8lzJ66Sv/iddc7LEoU8egOce2tA6WL9dZk3zr4SU4xH0RFg2yIdYq4ZL09NmKpv4PYvw3SmQ2MPXVsYw76Yamkhf4bXgjPNje2mwEzFvzXM93a1Lq/YdEWi0nQ8qTESM8B/pv8AUqdtgTJvIAx6/iXagsh+s2+mm9DZRZFD80SsSUnNWnjEPJCfuSrZ8uYYG40wFfbqKC6Ex0L24nPRrvccwDG7uzNzJhhSJ3Sm4MqGeX/5FsLA5MzJFDK6idQILyIU/6uJJILTWATv+BWgmvyWGsHPBe15JyvHQ1tPSguZcU04UzysBtW5VQMi/LERXAdBVkfVtH1YdTFJfxywVMlosDwVkSiWLSGFz6PSKaBTEy+yEFfi3uGxXdrtCmsxAwxg+bqedbAdnXpd03qE1br8Yalcr9StVd9Hkq+Jm9fGllqB2aSdDp1ByEvriBtEzwceAt/7X/At+ui2Nu2BLbtvtJKphM8SIxDW8sanAbWtkY+eqkQ93dv8kRnGeXXeZFxM+S1GlEfjlb2tt3Gc+t9xwVwCFI+NN8NnDtL3xtJrf6voxCia+VR/ARqfjQVkqut7H42O9+l87Mo2ZcAAmYHn+SEMaVYUr3QBrIWJ/w6rO6E9hLy95irK3L0Ls6N01MfS9Sl153yA+NPzjvU9EQ3ZWZYuPv3zZMmR+CwcAtPRMXLD+nFQ67GAT0Lkf7zsZ